# Ticket Storage (`fastgws` + `solvedrive`)

> Async `TicketStore` contract and implementations: `MemoryTicketStore`, `LocalTicketStore`, and Google Drive-backed `DriveTicketStore`.

All three stores enforce the same invariants:
- Raw browser HTML is **always** sanitized via `sanitize_dom` before persisting `dom.html`.
- Tickets live under a uniform folder layout:
  ```
  solveitcrafts/
    index.json
    <ticket_id>/
      ticket.json
      dom.html
      screenshot.png  (optional)
  ```

In [ ]:
#| default_exp drive

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import asyncio, json
from typing import Any
from fastcore.utils import *

from craft_support.models import CaptureMeta, DriveRefs, Ticket, TicketStatus, new_ticket_id
from craft_support.sanitize import SanitizeResult, sanitize_dom

DEFAULT_ROOT_FOLDER = "solveitcrafts"
INDEX_FILENAME = "index.json"
TICKET_FILENAME = "ticket.json"
DOM_FILENAME = "dom.html"
SCREENSHOT_FILENAME = "screenshot.png"

_VALID_TICKET_ID_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,127}$")

def validate_ticket_id(ticket_id: str) -> str:
    "Validate that `ticket_id` is a safe single path/query segment."
    if not isinstance(ticket_id, str) or ".." in ticket_id or not _VALID_TICKET_ID_RE.fullmatch(ticket_id):
        raise ValueError(f"Invalid ticket_id {ticket_id!r}")
    return ticket_id

def _escape_drive_q(value: str) -> str: return value.replace("\\", "\\\\").replace("'", "\\'")

def _ticket_from_index(meta: dict[str, Any]) -> Ticket:
    drive = meta.get("drive") or dict(folder_id=meta.get("folder_id"), folder_url=meta.get("folder_url"))
    return Ticket.from_dict({**meta, "drive": drive})

## `TicketStore` Contract

In [ ]:
#| export
class TicketNotFoundError(KeyError):
    "Raised when a ticket ID does not exist in the store."

def _new_ticket(tid: str, title: str, description: str, reporter_notes: str, capture: CaptureMeta | None, drive: DriveRefs) -> Ticket:
    return Ticket(id=tid, title=title.strip() or f'Bug report {tid}', description=description.strip(),
        reporter_notes=reporter_notes.strip(), capture=capture or CaptureMeta(), drive=drive)

class TicketStore:
    "Async ticket storage contract shared by `DriveTicketStore`, `LocalTicketStore`, and `MemoryTicketStore`."

    async def create_ticket(
        self,
        *,
        title: str, # Short ticket title
        description: str, # Detailed bug description
        reporter_notes: str = "", # Optional reporter reproduction notes
        capture: CaptureMeta | None = None, # Browser capture metadata
        raw_dom_html: str = "", # Raw HTML snapshot (always sanitized before persistence)
        screenshot_bytes: bytes | None = None, # Optional PNG screenshot bytes
        ticket_id: str | None = None, # Optional explicit ticket ID override
    ) -> tuple[Ticket, SanitizeResult]:
        "Sanitize DOM HTML and persist a new ticket bundle."
        raise NotImplementedError

    async def list_tickets(
        self,
        *,
        status: TicketStatus | None = None, # Optional lifecycle status filter
        refresh: bool = False, # Rebuild `index.json` by scanning ticket folders
    ) -> list[Ticket]:
        "List tickets, optionally filtered by lifecycle status."
        raise NotImplementedError

    async def get_ticket(self, ticket_id: str) -> Ticket: raise NotImplementedError

    async def update_ticket(self, ticket: Ticket) -> Ticket: raise NotImplementedError

    async def get_dom(self, ticket_id: str) -> str: raise NotImplementedError

    async def get_screenshot(self, ticket_id: str) -> bytes | None: raise NotImplementedError

    async def add_triage_note(self, ticket_id: str, text: str, author: str = "operator") -> Ticket:
        ticket = await self.get_ticket(ticket_id)
        ticket.add_note(text, author=author)
        return await self.update_ticket(ticket)

    async def set_status(self, ticket_id: str, status: TicketStatus) -> Ticket:
        ticket = await self.get_ticket(ticket_id)
        ticket.set_status(status)
        return await self.update_ticket(ticket)

In [ ]:
show_doc(TicketStore)

## `MemoryTicketStore`

Fast in-memory implementation used for deterministic tests and dry runs. Returns JSON-roundtripped copies so callers cannot mutate internal state accidentally.

In [ ]:
#| export
class MemoryTicketStore(TicketStore, BasicRepr):
    "In-memory async ticket store used for fast unit/integration tests and dry runs."
    def __init__(self, root_name: str = DEFAULT_ROOT_FOLDER) -> None:
        store_attr()
        self._tickets = {}
        self._doms = {}
        self._screenshots = {}
        self._index = {}

    async def create_ticket(
        self, *, title: str, description: str, reporter_notes: str = "", capture: CaptureMeta | None = None,
        raw_dom_html: str = "", screenshot_bytes: bytes | None = None, ticket_id: str | None = None,
    ) -> tuple[Ticket, SanitizeResult]:
        sanitized, tid = sanitize_dom(raw_dom_html), validate_ticket_id(ticket_id or new_ticket_id())
        refs = DriveRefs(
            folder_id=f"mem-folder-{tid}", folder_url=f"https://drive.google.com/drive/folders/mem-folder-{tid}",
            ticket_file_id=f"mem-ticket-{tid}",
            dom_file_id=f"mem-dom-{tid}" if sanitized.html else None,
            dom_url=f"https://drive.google.com/file/d/mem-dom-{tid}/view" if sanitized.html else None,
            screenshot_file_id=f"mem-shot-{tid}" if screenshot_bytes else None,
            screenshot_url=f"https://drive.google.com/file/d/mem-shot-{tid}/view" if screenshot_bytes else None)
        ticket = _new_ticket(tid, title, description, reporter_notes, capture, refs)
        self._tickets[tid] = Ticket.from_json(ticket.to_json())
        self._doms[tid] = sanitized.html
        if screenshot_bytes: self._screenshots[tid] = bytes(screenshot_bytes)
        self._index[tid] = ticket.index_entry()
        return Ticket.from_json(ticket.to_json()), sanitized

    async def list_tickets(self, *, status: TicketStatus | None = None, refresh: bool = False) -> list[Ticket]:
        items = [Ticket.from_json(t.to_json()) for t in self._tickets.values() if status is None or t.status == status]
        return sorted(items, key=lambda t: t.updated_at, reverse=True)

    async def get_ticket(self, ticket_id: str) -> Ticket:
        if ticket_id not in self._tickets: raise TicketNotFoundError(ticket_id)
        return Ticket.from_json(self._tickets[ticket_id].to_json())

    async def update_ticket(self, ticket: Ticket) -> Ticket:
        if ticket.id not in self._tickets: raise TicketNotFoundError(ticket.id)
        ticket.touch()
        self._tickets[ticket.id] = Ticket.from_json(ticket.to_json())
        self._index[ticket.id] = ticket.index_entry()
        return Ticket.from_json(ticket.to_json())

    async def get_dom(self, ticket_id: str) -> str:
        if ticket_id not in self._tickets: raise TicketNotFoundError(ticket_id)
        return self._doms.get(ticket_id, "")

    async def get_screenshot(self, ticket_id: str) -> bytes | None:
        if ticket_id not in self._tickets: raise TicketNotFoundError(ticket_id)
        return self._screenshots.get(ticket_id)

In [ ]:
import asyncio, tempfile
from fastcore.test import test_eq

In [ ]:
async def _test_memory_store():
    store = MemoryTicketStore()
    t, san = await store.create_ticket(title='Memory test', description='Checks CRUD',
        raw_dom_html='<script>bad()</script><input type="password" value="pw123"><p>Hello</p>', screenshot_bytes=b'\x89PNGfake')
    assert san.stripped_tags >= 1 and "pw123" not in await store.get_dom(t.id)
    test_eq(await store.get_screenshot(t.id), b"\x89PNGfake")
    await store.add_triage_note(t.id, "Triaged")
    test_eq((await store.get_ticket(t.id)).status, "triaged")
    await store.set_status(t.id, "closed")
    test_eq([x.id for x in await store.list_tickets(status="closed")], [t.id])

await _test_memory_store()

## `LocalTicketStore`

Filesystem-backed mirror of the exact Drive folder layout for offline development and local server/TUI runs (`--store local`).

In [ ]:
#| export
class LocalTicketStore(TicketStore, BasicRepr):
    "Filesystem-backed ticket store mirroring the exact Drive folder layout for local offline work."
    def __init__(self, root_dir: Path | str) -> None:
        self.root_dir = Path(root_dir)
        self.root_dir.mkdir(parents=True, exist_ok=True)
        self._index_lock = asyncio.Lock()

    def _index_path(self) -> Path: return self.root_dir/INDEX_FILENAME
    def _ticket_dir(self, ticket_id: str) -> Path:
        tid = validate_ticket_id(ticket_id)
        root = self.root_dir.resolve()
        target = (root/tid).resolve()
        if not target.is_relative_to(root) or target == root: raise ValueError(f"Invalid ticket_id {ticket_id!r}")
        return self.root_dir/tid

    def _load_index(self) -> dict[str, dict[str, Any]]:
        p = self._index_path()
        return p.read_json() if p.is_file() else {}

    def _save_index(self, idx: dict[str, dict]) -> None: self._index_path().write_json(idx, indent=2, ensure_ascii=False)

    async def create_ticket(
        self, *, title: str, description: str, reporter_notes: str = "", capture: CaptureMeta | None = None,
        raw_dom_html: str = "", screenshot_bytes: bytes | None = None, ticket_id: str | None = None,
    ) -> tuple[Ticket, SanitizeResult]:
        sanitized, tid = sanitize_dom(raw_dom_html), ticket_id or new_ticket_id()
        tdir = self._ticket_dir(tid)
        dom_path, shot_path, ticket_path = tdir/DOM_FILENAME, tdir/SCREENSHOT_FILENAME, tdir/TICKET_FILENAME
        dom_path.mk_write(sanitized.html, encoding="utf-8")
        if screenshot_bytes: shot_path.write_bytes(screenshot_bytes)
        refs = DriveRefs(folder_id=str(tdir), folder_url=tdir.resolve().as_uri(), ticket_file_id=str(ticket_path),
            dom_file_id=str(dom_path), dom_url=dom_path.resolve().as_uri(), screenshot_file_id=str(shot_path) if screenshot_bytes else None,
            screenshot_url=shot_path.resolve().as_uri() if screenshot_bytes else None)
        ticket = _new_ticket(tid, title, description, reporter_notes, capture, refs)
        ticket_path.write_text(ticket.to_json(), encoding="utf-8")
        async with self._index_lock:
            idx = self._load_index()
            idx[tid] = ticket.index_entry()
            self._save_index(idx)
        return ticket, sanitized

    async def list_tickets(self, *, status: TicketStatus | None = None, refresh: bool = False) -> list[Ticket]:
        if not refresh and self._index_path().is_file():
            idx = self._load_index()
            items = [_ticket_from_index(m) for m in idx.values() if status is None or m.get("status") == status]
            return sorted(items, key=lambda t: t.updated_at, reverse=True)
        tickets, idx = [], {}
        for child in sorted(self.root_dir.iterdir()):
            tfile = child/TICKET_FILENAME
            if not (child.is_dir() and tfile.is_file()): continue
            t = Ticket.from_json(tfile.read_text(encoding="utf-8"))
            idx[t.id] = t.index_entry()
            if status is None or t.status == status: tickets.append(t)
        async with self._index_lock: self._save_index(idx)
        return sorted(tickets, key=lambda t: t.updated_at, reverse=True)

    async def get_ticket(self, ticket_id: str) -> Ticket:
        tfile = self._ticket_dir(ticket_id)/TICKET_FILENAME
        if not tfile.is_file(): raise TicketNotFoundError(ticket_id)
        return Ticket.from_json(tfile.read_text(encoding="utf-8"))

    async def update_ticket(self, ticket: Ticket) -> Ticket:
        tfile = self._ticket_dir(ticket.id)/TICKET_FILENAME
        if not tfile.is_file(): raise TicketNotFoundError(ticket.id)
        ticket.touch()
        tfile.write_text(ticket.to_json(), encoding="utf-8")
        async with self._index_lock:
            idx = self._load_index()
            idx[ticket.id] = ticket.index_entry()
            self._save_index(idx)
        return ticket

    async def get_dom(self, ticket_id: str) -> str:
        dfile = self._ticket_dir(ticket_id)/DOM_FILENAME
        if not dfile.is_file(): raise TicketNotFoundError(ticket_id)
        return dfile.read_text(encoding="utf-8")

    async def get_screenshot(self, ticket_id: str) -> bytes | None:
        if not self._ticket_dir(ticket_id).is_dir(): raise TicketNotFoundError(ticket_id)
        sfile = self._ticket_dir(ticket_id)/SCREENSHOT_FILENAME
        return sfile.read_bytes() if sfile.is_file() else None

In [ ]:
async def _test_local_store():
    with tempfile.TemporaryDirectory() as td:
        store = LocalTicketStore(Path(td)/"tickets")
        t, _ = await store.create_ticket(title='Mobile drawer clipped', description='Overlaps bottom bar',
            raw_dom_html='<input type="password" value="hunter2"><p>Drawer</p>')
        loaded = await store.get_ticket(t.id)
        test_eq(loaded.title, "Mobile drawer clipped")
        assert "hunter2" not in await store.get_dom(t.id)
        assert (Path(td)/"tickets"/INDEX_FILENAME).is_file()
        try:
            await store.get_ticket("TKT-MISSING")
            raise AssertionError("Expected TicketNotFoundError")
        except TicketNotFoundError: pass
        try:
            await store.get_ticket("../escape")
            raise AssertionError("Expected ValueError for path traversal")
        except ValueError: pass

await _test_local_store()

## `DriveTicketStore`

Production store backed by `fastgws` OAuth credentials and `solvedrive.GDrive`. It maintains `solveitcrafts/index.json` so `list_tickets(status=...)` only hydrates matching folders unless `refresh=True` is requested.

In [ ]:
#| export
class DriveTicketStore(TicketStore, BasicRepr):
    "Google Workspace Drive ticket store powered by `fastgws` and `solvedrive`."
    def __init__(
        self,
        gd: Any, # `solvedrive.GDrive` client instance
        *,
        root_folder_name: str = DEFAULT_ROOT_FOLDER, # Top-level Drive folder name
    ) -> None:
        store_attr()
        self._root_folder = None
        self._index_lock = asyncio.Lock()

    @classmethod
    async def from_oauth(
        cls,
        *,
        account: str | None = None, # Google account email configured via `gclientid-auth`
        root_folder_name: str = DEFAULT_ROOT_FOLDER, # Top-level Drive folder name
        token_path: str | None = None, # Optional explicit OAuth token path
    ) -> DriveTicketStore:
        "Create a `DriveTicketStore` using `fastgws` OAuth credentials and `solvedrive.GDrive`."
        from fastgws import Drive as FGWSDrive
        from fastgws.auth import oauth_creds
        from solvedrive.core import GDrive

        creds = await oauth_creds(account=account, token_path=token_path)
        gd = GDrive(FGWSDrive(creds=creds)) if hasattr(GDrive, "init") else GDrive(creds)
        return cls(gd, root_folder_name=root_folder_name)

    async def ensure_root_folder(self) -> Any:
        "Locate or create the root Drive folder for tickets."
        from solvedrive.core import FOLDER_MIME
        if self._root_folder is not None: return self._root_folder
        q = f"name = '{_escape_drive_q(self.root_folder_name)}' and mimeType = '{FOLDER_MIME}' and trashed=false"
        found = await self.gd.search_files(q, max_results=5)
        self._root_folder = first(found) or await self.gd.create_folder(self.root_folder_name)
        return self._root_folder

### Drive Child & Index Helpers

Low-level helpers locate children by name, upsert media payloads in place via `fastgws` `files.update_media`, and read/write `solveitcrafts/index.json`.

In [ ]:
#| export
@patch
async def _find_child(self: DriveTicketStore, folder: Any, name: str) -> Any | None:
    return first(await folder.ls(q=f"name = '{_escape_drive_q(name)}'", max_results=5))

@patch
async def _write_media(self: DriveTicketStore, file_id: str, data: bytes, mime: str) -> None:
    await self.gd.api.files.update_media(file_id=file_id, media=data, media_type=mime, supports_all_drives=True)

@patch
async def _upsert_bytes(self: DriveTicketStore, folder: Any, name: str, data: bytes, mime: str) -> Any:
    if existing := await self._find_child(folder, name):
        await self._write_media(existing.id, data, mime)
        await existing.refresh()
        return existing
    return await folder.upload(data=data, name=name, mime=mime)

@patch
async def _load_index(self: DriveTicketStore, root: Any) -> dict[str, dict[str, Any]]:
    idx_file = await self._find_child(root, INDEX_FILENAME)
    if idx_file is None: return {}
    return loads((await idx_file.download()).decode("utf-8"))

@patch
async def _save_index(self: DriveTicketStore, root: Any, idx: dict[str, dict]) -> None:
    payload = json.dumps(idx, indent=2, ensure_ascii=False).encode("utf-8")
    await self._upsert_bytes(root, INDEX_FILENAME, payload, "application/json")

@patch
async def _find_ticket_folder(self: DriveTicketStore, ticket_id: str) -> Any:
    from solvedrive.core import FOLDER_MIME
    root = await self.ensure_root_folder()
    tid = validate_ticket_id(ticket_id)
    folder = first(await root.ls(q=f"name = '{_escape_drive_q(tid)}' and mimeType = '{FOLDER_MIME}'", max_results=5))
    if folder is None: raise TicketNotFoundError(ticket_id)
    return folder

### Ticket CRUD Operations

`create_ticket` always sanitizes raw HTML via `fast5ever` before uploading `dom.html`, records Drive file IDs back into `ticket.json`, and updates `index.json`.

In [ ]:
#| export
@patch
async def create_ticket(
    self: DriveTicketStore, *, title: str, description: str, reporter_notes: str = "", capture: CaptureMeta | None = None,
    raw_dom_html: str = "", screenshot_bytes: bytes | None = None, ticket_id: str | None = None,
) -> tuple[Ticket, SanitizeResult]:
    sanitized, tid = sanitize_dom(raw_dom_html), validate_ticket_id(ticket_id or new_ticket_id())
    root = await self.ensure_root_folder()
    folder = await self.gd.create_folder(tid, parent=root)

    dom_file = await folder.upload(data=sanitized.html.encode("utf-8"), name=DOM_FILENAME, mime="text/html")
    shot_file = await folder.upload(data=screenshot_bytes, name=SCREENSHOT_FILENAME, mime="image/png") if screenshot_bytes else None
    refs = DriveRefs(folder_id=folder.id, folder_url=folder.get('webViewLink'), dom_file_id=dom_file.id,
        dom_url=dom_file.get('webViewLink'), screenshot_file_id=shot_file.id if shot_file else None,
        screenshot_url=shot_file.get('webViewLink') if shot_file else None)
    ticket = _new_ticket(tid, title, description, reporter_notes, capture, refs)
    tfile = await folder.upload(data=ticket.to_json().encode("utf-8"), name=TICKET_FILENAME, mime="application/json")
    ticket.drive.ticket_file_id = tfile.id
    await self._write_media(tfile.id, ticket.to_json().encode("utf-8"), "application/json")

    async with self._index_lock:
        idx = await self._load_index(root)
        idx[tid] = ticket.index_entry()
        await self._save_index(root, idx)
    return ticket, sanitized

@patch
async def get_ticket(self: DriveTicketStore, ticket_id: str) -> Ticket:
    folder = await self._find_ticket_folder(ticket_id)
    tfile = await self._find_child(folder, TICKET_FILENAME)
    if tfile is None: raise TicketNotFoundError(ticket_id)
    ticket = Ticket.from_json(await tfile.download())
    ticket.drive.folder_id = ticket.drive.folder_id or folder.id
    ticket.drive.folder_url = ticket.drive.folder_url or folder.get("webViewLink")
    ticket.drive.ticket_file_id = tfile.id
    return ticket

@patch
async def list_tickets(self: DriveTicketStore, *, status: TicketStatus | None = None, refresh: bool = False) -> list[Ticket]:
    from solvedrive.core import FOLDER_MIME
    root = await self.ensure_root_folder()
    if not refresh and (idx := await self._load_index(root)):
        tickets = [_ticket_from_index(meta) for meta in idx.values() if status is None or meta.get("status") == status]
        return sorted(tickets, key=lambda t: t.updated_at, reverse=True)

    folders = await root.ls(q=f"mimeType = '{FOLDER_MIME}'", max_results=200)
    tickets, idx = [], {}
    for folder in folders:
        tfile = await self._find_child(folder, TICKET_FILENAME)
        if tfile is None: continue
        ticket = Ticket.from_json(await tfile.download())
        ticket.drive.folder_id = ticket.drive.folder_id or folder.id
        ticket.drive.folder_url = ticket.drive.folder_url or folder.get("webViewLink")
        ticket.drive.ticket_file_id = tfile.id
        idx[ticket.id] = ticket.index_entry()
        if status is None or ticket.status == status: tickets.append(ticket)
    async with self._index_lock: await self._save_index(root, idx)
    return sorted(tickets, key=lambda t: t.updated_at, reverse=True)

@patch
async def update_ticket(self: DriveTicketStore, ticket: Ticket) -> Ticket:
    root = await self.ensure_root_folder()
    ticket.touch()
    payload = ticket.to_json().encode("utf-8")
    if ticket.drive.ticket_file_id: await self._write_media(ticket.drive.ticket_file_id, payload, "application/json")
    else:
        folder = await self._find_ticket_folder(ticket.id)
        tfile = await self._upsert_bytes(folder, TICKET_FILENAME, payload, "application/json")
        ticket.drive.ticket_file_id = tfile.id
    async with self._index_lock:
        idx = await self._load_index(root)
        idx[ticket.id] = ticket.index_entry()
        await self._save_index(root, idx)
    return ticket

### Artifact Accessors (`get_dom`, `get_screenshot`)

In [ ]:
#| export
@patch
async def get_dom(self: DriveTicketStore, ticket_id: str) -> str:
    folder = await self._find_ticket_folder(ticket_id)
    dfile = await self._find_child(folder, DOM_FILENAME)
    return (await dfile.download()).decode("utf-8", errors="replace") if dfile else ""

@patch
async def get_screenshot(self: DriveTicketStore, ticket_id: str) -> bytes | None:
    folder = await self._find_ticket_folder(ticket_id)
    sfile = await self._find_child(folder, SCREENSHOT_FILENAME)
    return await sfile.download() if sfile else None

In [ ]:
show_doc(DriveTicketStore)

### Inline Test with `FakeGDrive`

We verify `DriveTicketStore` end-to-end offline using an in-memory double matching `solvedrive.GDrive` and `fastgws` `files.update_media`.

In [ ]:
#| hide
class _FakeDriveFile(AttrDict):
    def __init__(self, gd, fid, name, mime, data=b"", parents=None):
        super().__init__(id=fid, name=name, mimeType=mime, parents=parents or [], webViewLink=f"https://drive.google.com/file/d/{fid}/view")
        object.__setattr__(self, "_gd", gd)

    async def download(self): return self._gd._blobs[self.id]
    async def refresh(self): return self

    async def ls(self, q=None, **kwargs):
        children = [f for f in self._gd._files.values() if self.id in (f.get("parents") or [])]
        if q and "name = '" in q:
            target = q.split("name = '", 1)[1].split("'", 1)[0]
            children = [c for c in children if c.name == target]
        if q and "mimeType = '" in q:
            mime = q.split("mimeType = '", 1)[1].split("'", 1)[0]
            children = [c for c in children if c.mimeType == mime]
        return children

    async def upload(self, path=None, name=None, data=b"", mime="application/octet-stream"):
        return await self._gd.upload(name=name, data=data, mime=mime, folder=self)

class _FakeGDrive:
    def __init__(self):
        self._seq, self._files, self._blobs = 0, {}, {}
        self.api = AttrDict(files=AttrDict(update_media=self._update_media))

    def _new_id(self, prefix):
        self._seq += 1
        return f"{prefix}_{self._seq}"

    async def _update_media(self, *, file_id, media, media_type="", supports_all_drives=True):
        self._blobs[file_id] = bytes(media)
        return dict(id=file_id)

    async def search_files(self, q=None, max_results=20, **kwargs):
        items = list(self._files.values())
        if q and "name = '" in q:
            target = q.split("name = '", 1)[1].split("'", 1)[0]
            items = [i for i in items if i.name == target]
        if q and "mimeType = '" in q:
            mime = q.split("mimeType = '", 1)[1].split("'", 1)[0]
            items = [i for i in items if i.mimeType == mime]
        return items[:max_results]

    async def create_folder(self, name, parent=None):
        from solvedrive.core import FOLDER_MIME
        fid = self._new_id("fld")
        f = _FakeDriveFile(self, fid, name, FOLDER_MIME, parents=[parent.id] if parent else [])
        f["webViewLink"] = f"https://drive.google.com/drive/folders/{fid}"
        self._files[fid] = f
        return f

    async def upload(self, path=None, name=None, data=b"", mime="application/octet-stream", folder=None):
        fid = self._new_id("fil")
        f = _FakeDriveFile(self, fid, name or "file", mime, data=data, parents=[folder.id] if folder else [])
        self._files[fid] = f
        self._blobs[fid] = bytes(data)
        return f

async def _test_drive_store():
    store = DriveTicketStore(_FakeGDrive(), root_folder_name="solveitcrafts")
    ticket, san = await store.create_ticket(title='Inline editor focus lost', description='Pressing Enter loses focus',
        raw_dom_html='<script>evil()</script><input type="password" value="hunter2"><p>Editor</p>',
        screenshot_bytes=b'\x89PNG\r\n\x1a\nfakepng')
    assert ticket.status == "new" and ticket.drive.folder_id and ticket.drive.ticket_file_id
    dom = await store.get_dom(ticket.id)
    assert "<script" not in dom.lower() and "hunter2" not in dom and "Editor" in dom
    test_eq(await store.get_screenshot(ticket.id), b"\x89PNG\r\n\x1a\nfakepng")
    updated = await store.add_triage_note(ticket.id, "Confirmed in C2 editor")
    test_eq(updated.status, "triaged")
    test_eq([t.id for t in await store.list_tickets(status="triaged")], [ticket.id])
    test_eq(await store.list_tickets(status="new"), [])
    test_eq(len(await store.list_tickets(refresh=True)), 1)

await _test_drive_store()

### Live Google Drive Example

The cell below is tagged `#| live` so `nbdev-test` skips it during offline runs. Run it with `--flags live` when OAuth credentials are configured.

In [ ]:
#| live
async def _live_drive_check():
    store = await DriveTicketStore.from_oauth()
    tickets = await store.list_tickets()
    print(f"Found {len(tickets)} ticket(s) in Google Drive folder '{store.root_folder_name}'.")

await _live_drive_check()